In [1]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ── Colab / Environment Setup ──────────────────────────────────────────
import os, sys, glob
import numpy as np

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('SAIR'):
        os.system('git clone https://github.com/silvaxxx1/SAIR.git')
    # UTILS/ lives inside 5_GPT from scratch/
    sys.path.insert(0, 'SAIR/5_GPT from scratch/')
    DATA_DIR  = 'SAIR/5_GPT from scratch/data'
    BOOKS_DIR = 'SAIR/4_Applied Deep Learning with PyTorch/3_Sequence and NLP/harry_potter_txt/'
else:
    # CWD is already 5_GPT from scratch/ — UTILS/ is right here
    sys.path.insert(0, '.')
    DATA_DIR  = 'data'
    BOOKS_DIR = '../4_Applied Deep Learning with PyTorch/3_Sequence and NLP/harry_potter_txt/'

# ── Auto-generate .bin files if missing ────────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
train_bin = os.path.join(DATA_DIR, 'train_ids.bin')

if not os.path.exists(train_bin):
    print("Data files not found — generating from Harry Potter books...")
    import tiktoken

    book_files = sorted(glob.glob(os.path.join(BOOKS_DIR, '*.txt')))
    if not book_files:
        raise FileNotFoundError(
            f"No .txt books found in {BOOKS_DIR}\n"
            "Please make sure the Harry Potter books are in that directory."
        )

    full_text = '\n\n'.join(open(p, encoding='utf-8').read() for p in book_files)
    n = len(full_text)
    tokenizer_prep = tiktoken.encoding_for_model('gpt2')

    for split, text in [
        ('train', full_text[:int(n * 0.90)]),
        ('val',   full_text[int(n * 0.90):int(n * 0.97)]),
        ('test',  full_text[int(n * 0.97):]),
    ]:
        ids = np.array(tokenizer_prep.encode(text), dtype=np.int32)
        ids.tofile(os.path.join(DATA_DIR, f'{split}_ids.bin'))
        print(f"  {split:5s}: {len(ids):,} tokens saved")

    print("Done — data ready.")
else:
    print("Data files found — skipping generation.")

print(f'Running on : {"Colab" if IN_COLAB else "Local"}')
print(f'Data dir   : {DATA_DIR}')

Data files not found — generating from Harry Potter books...
  train: 1,740,472 tokens saved
  val  : 137,152 tokens saved
  test : 56,651 tokens saved
Done — data ready.
Running on : Colab
Data dir   : SAIR/5_GPT from scratch/data


In [3]:
class GPT2Dataset(Dataset):
    def __init__(self, file_path, max_length, stride):
        self.data       = np.fromfile(file_path, dtype=np.int32)
        self.max_length = max_length
        self.stride     = stride

    def __len__(self):
        return (len(self.data) - self.max_length) // self.stride

    def __getitem__(self, idx):
        start = idx * self.stride
        x = torch.tensor(self.data[start : start + self.max_length], dtype=torch.long)
        y = torch.tensor(self.data[start + 1 : start + self.max_length + 1], dtype=torch.long)
        return x, y


# Paths set by setup cell above — works on Colab and local
train_path = f"{DATA_DIR}/train_ids.bin"
val_path   = f"{DATA_DIR}/val_ids.bin"
test_path  = f"{DATA_DIR}/test_ids.bin"
# Note: run 1.DATA.ipynb first to generate these .bin files

MAX_LEN = 256
STRIDE  = 128
BATCH   = 4

torch.manual_seed(123)

train_dataset = GPT2Dataset(train_path, MAX_LEN, STRIDE)
val_dataset   = GPT2Dataset(val_path,   MAX_LEN, STRIDE)
test_dataset  = GPT2Dataset(test_path,  MAX_LEN, STRIDE)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, drop_last=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 3398
Val   batches : 267
Test  batches : 110


In [4]:
# sys.path already set in the setup cell above
from UTILS.model import GPTModel

# Reduced config — CPU friendly
GPT_CONFIG_CPU = {
    "vocab_size"     : 50257,
    "context_length" : 256,
    "emb_dim"        : 512,
    "n_heads"        : 8,
    "n_layers"       : 6,
    "drop_rate"      : 0.1,
    "qkv_bias"       : False,
}

# Full GPT-2 small — use on GPU
GPT_CONFIG_124M = {
    "vocab_size"     : 50257,
    "context_length" : 1024,
    "emb_dim"        : 768,
    "n_heads"        : 12,
    "n_layers"       : 12,
    "drop_rate"      : 0.1,
    "qkv_bias"       : False,
}

ACTIVE_CONFIG = GPT_CONFIG_CPU   # swap to GPT_CONFIG_124M when on GPU

model = GPTModel(ACTIVE_CONFIG)
model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Model parameters: 70,500,352


In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = F.cross_entropy(model(input_batch).flatten(0, 1), target_batch.flatten())
    return loss

In [ ]:
def trainerV0(model,single_batch,optimizer, device,num_epochs):

    input_batch , target_batch = single_batch 
    input_batch = input_batch.to(device)    
    target_batch = target_batch.to(device)    
    

    losses = [] 

    for epoch in range(num_epochs):
        model.train() 
        optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch , model , device)
        loss.backward() 
        optimizer.step() 
        losses.append(loss.item()) 

        if (epoch + 1) % 10 == 0 :
            print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")
        
        return losses

